# 파일별 ACF 및 FFT 분석

각 source_file의 시계열 자기상관과 주파수 성분을 탐색합니다.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
config = {
    "input_len": 400,
    "model_output_len": 100,
    "eval_output_len": 100,
    "stride": 100,
    "input_dim": 1,

    "hidden_size": 16,
    "num_layers": 2,

    "nhead": 2,
    "dim_feedforward": 64,
    "dropout": 0.0,

    "lr": 0.001,
    "epoch": 10000,
    "batch_size": 64,
    "grad_clip": 1.0,
    "patience": 10,
}

In [ ]:
!pip install numpy pandas matplotlib seaborn torch optuna scikit-learn statsmodels scipy seaborn

## 데이터 불러오기

분석에 사용할 원본 시계열 데이터를 읽고 기본 정보를 확인합니다.


## ACF 및 FFT 분석

파일별 주기성과 주요 주파수 성분을 계산합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import torch
import torch.nn as nn
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from torch.utils.data import DataLoader, TensorDataset, Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("combined_no_wave.csv")

df = df.rename(columns={
    '0': 'time',
    '1': 'surge',
    '2': 'sway',
    '3': 'heave',
    '4': 'roll',
    '5': 'pitch',
    '6': 'yaw',
    'Source_File': 'source_file'
})

numeric_cols = ['time', 'surge', 'sway', 'heave', 'roll', 'pitch', 'yaw']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=numeric_cols)

print("shape:", df.shape)
print(df.dtypes)
print(df.head())

time = df['time'].values
heave = df['heave'].values

dt = time[1] - time[0]
fs = 1 / dt

print(f"샘플 수: {len(heave)}")
print(f"총 길이: {time[-1] - time[0]:.2f} s")
print(f"샘플링 간격 dt: {dt:.4f} s")
print(f"샘플링 주파수 fs: {fs:.4f} Hz")

plt.figure(figsize=(12, 4))
plt.plot(time, heave)
plt.xlabel("Time (s)")
plt.ylabel("Heave")
plt.title("Raw Heave Signal")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

time = df["time"].astype(float).values
heave = df["heave"].astype(float).values

dt = np.mean(np.diff(time))
fs = 1.0 / dt

print("── 데이터 기본 정보 ──")
print(f"샘플 수: {len(heave)}")
print(f"총 길이: {time[-1] - time[0]:.2f} s")
print(f"샘플링 간격 dt: {dt:.4f} s")
print(f"샘플링 주파수 fs: {fs:.4f} Hz")

In [ ]:
'''
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(12, 4))
plt.plot(time, heave)
plt.xlabel("Time (s)")
plt.ylabel("Heave")
plt.title("Raw Heave Signal")
plt.grid(True)
plt.show()


fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(heave, lags=300, ax=ax)
ax.set_title("ACF Analysis for Heave Data")
ax.set_xlabel("Lags")
ax.set_ylabel("Autocorrelation")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


dt = 0.1

heave_centered = heave - np.mean(heave)

fft_vals = np.abs(np.fft.fft(heave_centered))
freq = np.fft.fftfreq(len(heave_centered), d=dt)

mask = freq > 0
freq = freq[mask]
fft_vals = fft_vals[mask]

dominant_freq = freq[np.argmax(fft_vals)]
period_sec = 1 / dominant_freq

plt.figure(figsize=(10, 4))
plt.plot(freq, fft_vals, color='royalblue')

plt.xlim(0, 1.0)
plt.axvline(x=dominant_freq, color='red', linestyle='--', alpha=0.6)
plt.text(
    dominant_freq + 0.02,
    max(fft_vals) * 0.9,
    f'Peak: {dominant_freq:.4f}Hz\n({period_sec:.2f}s)',
    color='red',
    fontweight='bold'
)

plt.title('FFT Frequency Spectrum (Focus on Low Frequency)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Dominant Frequency: {dominant_freq:.6f} Hz')
print(f'Estimated Period: {period_sec:.2f} seconds')


result = adfuller(heave)

print("── ADF 정상성 검정 ──")
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")

if result[1] < 0.05:
    print("결론: Stationary, 정상성 있음")
else:
    print("결론: Non-stationary, 정상성 없음")
'''

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import acf
from scipy.signal import find_peaks


DT = 0.1
MAX_LAG = 300

MIN_PEAK_LAG = 30

MIN_PEAK_HEIGHT = 0.1


acf_results = []
acf_curves = []

for source_file, group in df.groupby("source_file", sort=False):

    group = group.sort_values("time")

    heave_file = (
        pd.to_numeric(group["heave"], errors="coerce")
        .dropna()
        .to_numpy()
    )

    if len(heave_file) < MAX_LAG + 10:
        print(
            f"[SKIP] {source_file}: "
            f"데이터 길이 부족 ({len(heave_file)} samples)"
        )
        continue

    heave_centered = heave_file - np.mean(heave_file)

    max_lag_for_file = min(MAX_LAG, len(heave_centered) - 1)

    acf_values = acf(
        heave_centered,
        nlags=max_lag_for_file,
        fft=True,
    )

    lags = np.arange(len(acf_values))

    search_acf = acf_values.copy()
    search_acf[:MIN_PEAK_LAG] = -np.inf

    peak_indices, peak_properties = find_peaks(
        search_acf,
        height=MIN_PEAK_HEIGHT,
        distance=20,
    )

    if len(peak_indices) == 0:
        print(f"[WARN] {source_file}: 유의미한 ACF peak 없음")

        acf_results.append({
            "source_file": source_file,
            "num_samples": len(heave_file),
            "first_peak_lag": np.nan,
            "first_peak_period_sec": np.nan,
            "first_peak_acf": np.nan,
            "strongest_peak_lag": np.nan,
            "strongest_peak_period_sec": np.nan,
            "strongest_peak_acf": np.nan,
        })

    else:
        first_peak_lag = int(peak_indices[0])
        first_peak_acf = float(acf_values[first_peak_lag])

        strongest_idx = peak_indices[
            np.argmax(acf_values[peak_indices])
        ]
        strongest_peak_lag = int(strongest_idx)
        strongest_peak_acf = float(acf_values[strongest_peak_lag])

        acf_results.append({
            "source_file": source_file,
            "num_samples": len(heave_file),

            "first_peak_lag": first_peak_lag,
            "first_peak_period_sec": first_peak_lag * DT,
            "first_peak_acf": first_peak_acf,

            "strongest_peak_lag": strongest_peak_lag,
            "strongest_peak_period_sec": strongest_peak_lag * DT,
            "strongest_peak_acf": strongest_peak_acf,
        })

    acf_curves.append({
        "source_file": source_file,
        "lags": lags,
        "acf": acf_values,
    })


acf_df = pd.DataFrame(acf_results)

display(acf_df)

valid_first = acf_df.dropna(subset=["first_peak_lag"])
valid_strongest = acf_df.dropna(subset=["strongest_peak_lag"])

print("\n── source_file별 ACF 요약 ──")
print(f"분석 파일 수: {len(acf_df)}")
print(f"유효 peak 파일 수: {len(valid_first)}")

if len(valid_first) > 0:
    print(
        f"첫 번째 peak lag 평균: "
        f"{valid_first['first_peak_lag'].mean():.2f}"
    )
    print(
        f"첫 번째 peak lag 중앙값: "
        f"{valid_first['first_peak_lag'].median():.2f}"
    )
    print(
        f"첫 번째 peak period 평균: "
        f"{valid_first['first_peak_period_sec'].mean():.2f} s"
    )
    print(
        f"첫 번째 peak period 중앙값: "
        f"{valid_first['first_peak_period_sec'].median():.2f} s"
    )

if len(valid_strongest) > 0:
    print(
        f"가장 강한 peak lag 평균: "
        f"{valid_strongest['strongest_peak_lag'].mean():.2f}"
    )
    print(
        f"가장 강한 peak period 평균: "
        f"{valid_strongest['strongest_peak_period_sec'].mean():.2f} s"
    )


acf_df.to_csv(
    "acf_results_by_source_file.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n저장 완료: acf_results_by_source_file.csv")


plt.figure(figsize=(12, 5))

for item in acf_curves:
    plt.plot(
        item["lags"],
        item["acf"],
        alpha=0.2,
    )

plt.axhline(0, color="black", linewidth=1)
plt.axvline(100, color="red", linestyle="--", alpha=0.6, label="lag 100")
plt.axvline(113, color="orange", linestyle="--", alpha=0.6, label="lag 113")

plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.title("ACF of Heave by Source File")
plt.xlim(0, MAX_LAG)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


if len(valid_first) > 0:
    plt.figure(figsize=(10, 4))

    plt.hist(
        valid_first["first_peak_lag"],
        bins=min(20, len(valid_first)),
        edgecolor="black",
    )

    plt.axvline(
        valid_first["first_peak_lag"].mean(),
        linestyle="--",
        label=(
            f"Mean: "
            f"{valid_first['first_peak_lag'].mean():.1f}"
        ),
    )

    plt.xlabel("First ACF Peak Lag")
    plt.ylabel("Number of Source Files")
    plt.title("Distribution of First ACF Peak Lag")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


if len(valid_first) > 0:
    plt.figure(figsize=(10, 4))

    plt.hist(
        valid_first["first_peak_period_sec"],
        bins=min(20, len(valid_first)),
        edgecolor="black",
    )

    plt.xlabel("First ACF Peak Period (s)")
    plt.ylabel("Number of Source Files")
    plt.title("Distribution of First ACF Peak Period")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


FFT_DT = 0.1  # 샘플링 간격이 실제로 0.1초라면 유지
FFT_MAX_FREQ = 1.0

fft_results = []
spectra = []

for source_file, group in df.groupby("source_file", sort=False):
    group = group.sort_values("time")

    heave_file = (
        pd.to_numeric(group["heave"], errors="coerce")
        .dropna()
        .to_numpy()
    )

    if len(heave_file) < 10:
        print(f"[SKIP] {source_file}: 데이터 길이 부족")
        continue

    heave_centered = heave_file - np.mean(heave_file)

    fft_amplitude = np.abs(np.fft.rfft(heave_centered))
    frequencies = np.fft.rfftfreq(
        len(heave_centered),
        d=FFT_DT,
    )

    valid = frequencies > 0
    frequencies = frequencies[valid]
    fft_amplitude = fft_amplitude[valid]

    frequency_range = frequencies <= FFT_MAX_FREQ

    frequencies_focus = frequencies[frequency_range]
    fft_focus = fft_amplitude[frequency_range]

    if len(frequencies_focus) == 0:
        print(f"[SKIP] {source_file}: 주파수 구간 없음")
        continue

    dominant_index = np.argmax(fft_focus)
    dominant_frequency = frequencies_focus[dominant_index]
    dominant_period = 1.0 / dominant_frequency

    fft_results.append({
        "source_file": source_file,
        "num_samples": len(heave_file),
        "duration_sec": len(heave_file) * FFT_DT,
        "dominant_frequency_hz": dominant_frequency,
        "dominant_period_sec": dominant_period,
        "heave_mean": np.mean(heave_file),
        "heave_std": np.std(heave_file),
        "heave_peak_to_peak": np.ptp(heave_file),
    })

    normalized_fft = fft_focus / (np.max(fft_focus) + 1e-12)

    spectra.append({
        "source_file": source_file,
        "frequency": frequencies_focus,
        "normalized_amplitude": normalized_fft,
    })


fft_df = pd.DataFrame(fft_results)

if fft_df.empty:
    raise RuntimeError("FFT 분석 가능한 source_file이 없습니다.")

display(fft_df)

print("\n── 파일별 FFT 요약 ──")
print(f"분석 파일 수: {len(fft_df)}")
print(
    f"Dominant frequency 평균: "
    f"{fft_df['dominant_frequency_hz'].mean():.6f} Hz"
)
print(
    f"Dominant frequency 표준편차: "
    f"{fft_df['dominant_frequency_hz'].std():.6f} Hz"
)
print(
    f"Dominant period 평균: "
    f"{fft_df['dominant_period_sec'].mean():.2f} s"
)
print(
    f"Dominant period 중앙값: "
    f"{fft_df['dominant_period_sec'].median():.2f} s"
)

fft_df.to_csv(
    "fft_results_by_source_file.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n저장 완료: fft_results_by_source_file.csv")


plt.figure(figsize=(12, 5))

for spectrum in spectra:
    plt.plot(
        spectrum["frequency"],
        spectrum["normalized_amplitude"],
        alpha=0.25,
    )

plt.xlabel("Frequency (Hz)")
plt.ylabel("Normalized amplitude")
plt.title("Normalized Heave FFT Spectrum by Source File")
plt.xlim(0, FFT_MAX_FREQ)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 4))
plt.hist(
    fft_df["dominant_frequency_hz"],
    bins=min(20, len(fft_df)),
    edgecolor="black",
)
plt.xlabel("Dominant frequency (Hz)")
plt.ylabel("Number of source files")
plt.title("Distribution of Dominant Heave Frequency")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 4))
plt.hist(
    fft_df["dominant_period_sec"],
    bins=min(20, len(fft_df)),
    edgecolor="black",
)
plt.xlabel("Dominant period (s)")
plt.ylabel("Number of source files")
plt.title("Distribution of Dominant Heave Period")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
acf_fft_results = []

for source_file, group in df.groupby("source_file", sort=False):
    group = group.sort_values("time")

    heave_file = (
        pd.to_numeric(group["heave"], errors="coerce")
        .dropna()
        .to_numpy()
    )

    if len(heave_file) < 300:
        continue

    row = fft_df[fft_df["source_file"] == source_file]

    if row.empty:
        continue

    fft_period_sec = float(row.iloc[0]["dominant_period_sec"])
    expected_lag = int(round(fft_period_sec / DT))

    heave_centered = heave_file - np.mean(heave_file)

    max_lag = min(300, len(heave_centered) - 1)

    acf_values = acf(
        heave_centered,
        nlags=max_lag,
        fft=True,
    )

    search_start = max(1, int(expected_lag * 0.7))
    search_end = min(max_lag, int(expected_lag * 1.3))

    search_segment = acf_values[search_start:search_end + 1]

    best_lag = search_start + int(np.argmax(search_segment))
    best_acf = float(acf_values[best_lag])

    acf_fft_results.append({
        "source_file": source_file,
        "fft_period_sec": fft_period_sec,
        "fft_expected_lag": expected_lag,
        "acf_peak_lag": best_lag,
        "acf_period_sec": best_lag * DT,
        "acf_peak_value": best_acf,
        "period_difference_sec": abs(
            best_lag * DT - fft_period_sec
        ),
    })

acf_fft_df = pd.DataFrame(acf_fft_results)

display(acf_fft_df)

print("── FFT-ACF 비교 ──")
print(
    f"FFT 예상 lag 평균: "
    f"{acf_fft_df['fft_expected_lag'].mean():.2f}"
)
print(
    f"ACF peak lag 평균: "
    f"{acf_fft_df['acf_peak_lag'].mean():.2f}"
)
print(
    f"ACF peak lag 중앙값: "
    f"{acf_fft_df['acf_peak_lag'].median():.2f}"
)
print(
    f"FFT-ACF 주기 차이 평균: "
    f"{acf_fft_df['period_difference_sec'].mean():.2f} s"
)
print(
    f"ACF peak 상관계수 평균: "
    f"{acf_fft_df['acf_peak_value'].mean():.3f}"
)